# Tool-Router Ladder: teacher-forced generation (zero-shot and SFT)

**Settings:** GPU T4, Internet on, the dataset attached, and secret `HF_TOKEN` ticked.

The Mac renders every prompt and scores every completion. This notebook only maps prompts to completions.

**Download afterwards:** each `completions-*.jsonl` and its `.meta.json` from `/kaggle/working/out/`. Put them in `results/` locally.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN"])

In [ ]:
!pip install -q vllm

!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "CHANGE-ME"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

OUT = '/kaggle/working/out'
os.makedirs(OUT, exist_ok=True)

### Zero-shot (for Gate 2)

The third run is the review's handicap check: the same 1.5B model, but with tools rendered the way Qwen's own template does.

In [ ]:
!python src/generate.py --config configs/qwen05b.yaml --prompts "$DATA/prompts-c16.jsonl" --out "$OUT/completions-qwen05b-zeroshot-c16.jsonl"
!python src/generate.py --config configs/qwen15b.yaml --prompts "$DATA/prompts-c16.jsonl" --out "$OUT/completions-qwen15b-zeroshot-c16.jsonl"
!python src/generate.py --config configs/qwen15b.yaml --prompts "$DATA/prompts-c16-json.jsonl" --out "$OUT/completions-qwen15b-zeroshot-c16-json.jsonl"

### Fine-tuned students (after the training notebook has pushed both adapters)

In [ ]:
!python src/generate.py --config configs/qwen05b.yaml --adapter "$HF_USER/tool-router-qwen05b-sft" --prompts "$DATA/prompts-c16.jsonl" --out "$OUT/completions-qwen05b-sft-c16.jsonl"
!python src/generate.py --config configs/qwen15b.yaml --adapter "$HF_USER/tool-router-qwen15b-sft" --prompts "$DATA/prompts-c16.jsonl" --out "$OUT/completions-qwen15b-sft-c16.jsonl"

### Later: catalogs 40 and 80 for the scaling grid

Run the same commands with `prompts-c40.jsonl` / `prompts-c80.jsonl` and `-c40` / `-c80` in the output name.

If vLLM fails with an adapter, add `--backend hf`.